# Task 1: Data Preparation

### Objective
Prepare text data for GPT training.

- Load dataset
- Convert to lowercase
- Remove unwanted symbols
- Tokenize text
- Build vocabulary
- Create input-output sequences

Example:

- Input: 'the cat chased'

- Target: 'the'



In [ ]:
texts = [
    "The cat chased the mouse.",
    "The dog barked loudly.",
    "Machine learning is powerful.",
    "Deep learning uses neural networks."
]


In [ ]:
import re
import tensorflow as tf

In [ ]:
texts = [re.sub(r'[^a-zA-Z\s]', '', t.lower()) for t in texts]

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
vocab_size = len(tokenizer.word_index) + 1
print('Vocab size:', vocab_size)
print('Sequences:', sequences)

Vocab size: 16
Sequences: [[1, 3, 4, 1, 5], [1, 6, 7, 8], [9, 2, 10, 11], [12, 2, 13, 14, 15]]


## **Task 2: Token Embeddings**

**Objective**

Convert token IDs into dense vectors.

Example:

- cat → \[0.12,0.55,0.34\]  
- dog → \[0.21,0.77,0.61\]

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Correcting vocab_size based on the output of the previous cell (9gafXkAbmfl4)
vocab_size = 16
embedding_dim = 4

embedding_layer = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim)

# Pad the sequences to ensure they are all the same length
max_sequence_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

input_ids = tf.constant(padded_sequences)
dense_vectors = embedding_layer(input_ids)

print("Output shape:", dense_vectors.shape)
print("Dense Vectors:\n", dense_vectors.numpy())

Output shape: (4, 5, 4)
Dense Vectors:
 [[[ 0.00311838  0.02067086  0.01304718  0.00829403]
  [ 0.03618798  0.0127051   0.0141942  -0.0359697 ]
  [-0.01359724  0.03684363 -0.03136909  0.0096624 ]
  [ 0.00311838  0.02067086  0.01304718  0.00829403]
  [ 0.04898455 -0.02951225 -0.0175627   0.01077534]]

 [[ 0.00311838  0.02067086  0.01304718  0.00829403]
  [-0.02749931 -0.03766518  0.00542725 -0.03992627]
  [ 0.00729658  0.03571731 -0.00310017  0.01009183]
  [ 0.00045569 -0.01380844  0.02149932  0.02872627]
  [-0.01398333  0.04035112  0.03472977 -0.04068078]]

 [[-0.0415622   0.03475719 -0.00460768  0.02154536]
  [-0.01406764 -0.02484863 -0.03996854  0.03021778]
  [-0.03007189 -0.03061926 -0.0305946  -0.01936117]
  [ 0.04247654 -0.00259035  0.03100935 -0.04238088]
  [-0.01398333  0.04035112  0.03472977 -0.04068078]]

 [[ 0.03832174 -0.0299868   0.04970414  0.03198362]
  [-0.01406764 -0.02484863 -0.03996854  0.03021778]
  [ 0.02960838 -0.02538228 -0.04008698  0.01090489]
  [-0.00035473 -0.

## **Task 3: Positional Encoding**

**Objective**

Add positional information.

Example:

- the → position 0  
- cat → position 1  
- chased → position 2

In [ ]:
import numpy as np

# Retrieve necessary variables from the previous cells (kernel state)
# max_sequence_length = 5 (from BVbT5168niu8)
# embedding_dim = 4 (updated in BVbT5168niu8)
# dense_vectors (from BVbT5168niu8)

# Create positional encoding matrix
positional_encoding = np.zeros((max_sequence_length, embedding_dim))
for pos in range(max_sequence_length):
    for i in range(embedding_dim):
        if i % 2 == 0:  # Even dimensions
            positional_encoding[pos, i] = np.sin(pos / (10000 ** (i / embedding_dim)))
        else:  # Odd dimensions
            positional_encoding[pos, i] = np.cos(pos / (10000 ** ((i - 1) / embedding_dim)))

# Convert to TensorFlow tensor
positional_encoding = tf.constant(positional_encoding, dtype=tf.float32)

# Add positional encoding to dense vectors
# Ensure positional_encoding has a batch dimension for broadcasting if dense_vectors has one
# dense_vectors shape: (batch_size, sequence_length, embedding_dim)
# positional_encoding shape: (sequence_length, embedding_dim)

# Add an extra dimension to positional_encoding to enable broadcasting across the batch size
positional_encoding = tf.expand_dims(positional_encoding, axis=0) # Shape becomes (1, max_sequence_length, embedding_dim)

encoded_vectors = dense_vectors + positional_encoding

print("Positional Encoding shape:", positional_encoding.shape)
print("Dense Vectors with Positional Encoding shape:", encoded_vectors.shape)
print("Dense Vectors with Positional Encoding:\n", encoded_vectors.numpy())

Positional Encoding shape: (1, 5, 4)
Dense Vectors with Positional Encoding shape: (4, 5, 4)
Dense Vectors with Positional Encoding:
 [[[ 0.00311838  1.0206709   0.01304718  1.008294  ]
  [ 0.87765896  0.55300736  0.02419404  0.9639803 ]
  [ 0.89570016 -0.37930322 -0.01137042  1.0094625 ]
  [ 0.14423838 -0.9693216   0.04304269  1.0078441 ]
  [-0.707818   -0.68315583  0.02242664  1.0099754 ]]

 [[ 0.00311838  1.0206709   0.01304718  1.008294  ]
  [ 0.81397164  0.5026371   0.01542709  0.9600237 ]
  [ 0.91659397 -0.38042954  0.0168985   1.0098919 ]
  [ 0.1415757  -1.003801    0.05149483  1.0282763 ]
  [-0.7707858  -0.6132925   0.0747191   0.95851934]]

 [[-0.0415622   1.0347571  -0.00460768  1.0215454 ]
  [ 0.8274033   0.51545364 -0.02996871  1.0301678 ]
  [ 0.8792255  -0.4467661  -0.01059593  0.9804388 ]
  [ 0.18359654 -0.99258286  0.06100485  0.9571692 ]
  [-0.7707858  -0.6132925   0.0747191   0.95851934]]

 [[ 0.03832174  0.9700132   0.04970414  1.0319836 ]
  [ 0.8274033   0.51545364 -

## **Task 4: Masked Self Attention**

**Objective**

Implement causal attention.

Example:

- Current token:

- cat

Allowed:

- the  
- cat

Not Allowed:

- chased  
- mouse

* * *

**Deliverables**

Display:

- Attention Mask  
- Attention Scores  
- Attention Weights

* * *

## **Task 5: Multi Head Attention**

Implement:

4 Attention Heads

Analyze:

- Head 1 → Grammar  
- Head 2 → Context  
- Head 3 → Long Dependencies  
- Head 4 → Semantics

In [ ]:
import tensorflow as tf

# Assuming `encoded_vectors`, `embedding_dim`, `max_sequence_length` are available from previous cells
# encoded_vectors shape: (batch_size, max_sequence_length, embedding_dim)

# Linear layers for Query, Key, Value projections
query_layer = tf.keras.layers.Dense(embedding_dim, use_bias=False, name='query_projection')
key_layer = tf.keras.layers.Dense(embedding_dim, use_bias=False, name='key_projection')
value_layer = tf.keras.layers.Dense(embedding_dim, use_bias=False, name='value_projection')

# Project encoded vectors to Q, K, V
query = query_layer(encoded_vectors) # (batch_size, seq_len, embedding_dim)
key = key_layer(encoded_vectors)     # (batch_size, seq_len, embedding_dim)
value = value_layer(encoded_vectors) # (batch_size, seq_len, embedding_dim)

# 1. Calculate Attention Scores (Q * K^T)
# (batch_size, seq_len, embedding_dim) @ (batch_size, embedding_dim, seq_len) -> (batch_size, seq_len, seq_len)
attention_scores = tf.matmul(query, key, transpose_b=True)

# Scale the attention scores (divide by sqrt(embedding_dim))
scaled_attention_scores = attention_scores / tf.math.sqrt(tf.cast(embedding_dim, tf.float32))

# 2. Create Causal (Look-Ahead) Mask
# Mask future tokens from being attended to.
# We want to create a lower triangular matrix of ones.
# tf.ones_like creates a matrix of ones with the same shape as scaled_attention_scores.
# tf.linalg.band_part(matrix, num_lower, num_upper)
# num_lower = -1 for all lower triangular parts
# num_upper = 0 for no upper triangular parts (including diagonal)
causal_mask = tf.linalg.band_part(tf.ones_like(scaled_attention_scores), -1, 0)

# Apply the mask
# Replace upper triangular parts (future tokens) with a very small negative number (-1e9)
# so they become 0 after softmax.
masked_attention_scores = scaled_attention_scores + (1 - causal_mask) * -1e9

# 3. Calculate Attention Weights (softmax over scores)
attention_weights = tf.nn.softmax(masked_attention_scores, axis=-1)

# 4. Calculate Attention Output (Attention Weights * V)
# (batch_size, seq_len, seq_len) @ (batch_size, seq_len, embedding_dim) -> (batch_size, seq_len, embedding_dim)
attention_output = tf.matmul(attention_weights, value)

print("--- Task 4: Masked Self Attention ---")
print("Attention Mask (causal_mask):\n", causal_mask.numpy()[0]) # Showing for first batch item
print("Attention Scores (scaled_attention_scores):\n", scaled_attention_scores.numpy()[0]) # Showing for first batch item
print("Masked Attention Scores:\n", masked_attention_scores.numpy()[0]) # Showing for first batch item
print("Attention Weights (normalized scores):\n", attention_weights.numpy()[0]) # Showing for first batch item
print("Attention Output shape:", attention_output.shape)
print("Attention Output (first batch item):\n", attention_output.numpy()[0])

--- Task 4: Masked Self Attention ---
Attention Mask (causal_mask):
 [[1. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0.]
 [1. 1. 1. 0. 0.]
 [1. 1. 1. 1. 0.]
 [1. 1. 1. 1. 1.]]
Attention Scores (scaled_attention_scores):
 [[ 0.12640384 -0.3108944  -0.49041665 -0.25009456  0.27047598]
 [ 0.02176707 -0.12182602 -0.18757024 -0.01562707  0.10492016]
 [-0.1318988   0.20474118  0.34642708  0.281918   -0.17725976]
 [-0.13029166  0.37706453  0.5638469   0.2828638  -0.33197823]
 [-0.03928172  0.1520185   0.23880003  0.05063827 -0.13090312]]
Masked Attention Scores:
 [[ 1.26403838e-01 -1.00000000e+09 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09]
 [ 2.17670668e-02 -1.21826015e-01 -1.00000000e+09 -1.00000000e+09
  -1.00000000e+09]
 [-1.31898805e-01  2.04741180e-01  3.46427083e-01 -1.00000000e+09
  -1.00000000e+09]
 [-1.30291656e-01  3.77064526e-01  5.63846886e-01  2.82863796e-01
  -1.00000000e+09]
 [-3.92817222e-02  1.52018502e-01  2.38800034e-01  5.06382696e-02
  -1.30903125e-01]]
Attention Weights (normaliz

In [ ]:
import tensorflow as tf

# Assuming `encoded_vectors`, `embedding_dim`, `max_sequence_length` are available
# encoded_vectors shape: (batch_size, max_sequence_length, embedding_dim)

num_heads = 4
depth = embedding_dim // num_heads # Dimension of each head's output

# Ensure embedding_dim is divisible by num_heads
if embedding_dim % num_heads != 0:
    raise ValueError("embedding_dim must be divisible by num_heads")

class MultiHeadSelfAttention(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim

        assert embedding_dim % self.num_heads == 0

        self.depth = embedding_dim // self.num_heads

        self.wq = tf.keras.layers.Dense(embedding_dim, use_bias=False)
        self.wk = tf.keras.layers.Dense(embedding_dim, use_bias=False)
        self.wv = tf.keras.layers.Dense(embedding_dim, use_bias=False)

        self.dense = tf.keras.layers.Dense(embedding_dim)

    def split_heads(self, x, batch_size):
        # x shape: (batch_size, seq_len, embedding_dim)
        # Split the last dimension into (num_heads, depth).
        # Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, v, k, q, mask):
        batch_size = tf.shape(q)[0]

        q = self.wq(q)  # (batch_size, seq_len, embedding_dim)
        k = self.wk(k)  # (batch_size, seq_len, embedding_dim)
        v = self.wv(v)  # (batch_size, seq_len, embedding_dim)

        q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
        k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
        v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

        # Scaled dot-product attention
        # (batch_size, num_heads, seq_len_q, depth) * (batch_size, num_heads, depth, seq_len_k)
        # -> attention_scores.shape == (batch_size, num_heads, seq_len_q, seq_len_k)
        matmul_qk = tf.matmul(q, k, transpose_b=True)

        # scale matmul_qk
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        # add the mask to the scaled_attention_logits.
        if mask is not None:
            # mask needs to be broadcastable. Assuming mask is (batch_size, 1, seq_len_q, seq_len_k)
            scaled_attention_logits += (mask * -1e9)

        # softmax is normalized on the last axis (seq_len_k) so that the scores
        # add up to 1.
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)

        output = tf.matmul(attention_weights, v)  # (batch_size, num_heads, seq_len_q, depth)

        # Transpose back to (batch_size, seq_len_q, num_heads, depth)
        output = tf.transpose(output, perm=[0, 2, 1, 3])

        # Concat heads
        concat_attention = tf.reshape(output, (batch_size, -1, self.embedding_dim))  # (batch_size, seq_len_q, embedding_dim)

        output = self.dense(concat_attention)  # (batch_size, seq_len_q, embedding_dim)

        return output, attention_weights, scaled_attention_logits # Return logits for individual head analysis

# Create a causal mask for multi-head attention
def create_look_ahead_mask(seq_len):
    mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    return mask # (seq_len, seq_len)

# Generate the mask for the current max_sequence_length
look_ahead_mask = create_look_ahead_mask(max_sequence_length)
# Add batch and head dimensions for broadcasting with scaled_attention_logits (batch_size, num_heads, seq_len, seq_len)
look_ahead_mask = tf.expand_dims(tf.expand_dims(look_ahead_mask, axis=0), axis=0) # (1, 1, seq_len, seq_len)

# Initialize and call MultiHeadSelfAttention layer
mha_layer = MultiHeadSelfAttention(embedding_dim, num_heads)
multi_head_output, multi_head_attention_weights, all_head_logits = mha_layer(encoded_vectors, encoded_vectors, encoded_vectors, look_ahead_mask)

print("--- Task 5: Multi Head Attention ---")
print("Input (encoded_vectors) shape:", encoded_vectors.shape)
print("Multi-Head Attention Output shape:", multi_head_output.shape)
print("Multi-Head Attention Weights shape (per head, before concat):", multi_head_attention_weights.shape)

# Display attention weights for each head (for the first batch item, first query token)
print("\nAttention Weights for First Token (all heads):\n")
for i in range(num_heads):
    print(f"Head {i+1} Attention Weights (first batch item, first query token):\n", multi_head_attention_weights[0, i, 0].numpy())

# For analysis, let's display the raw (masked) scores for each head before softmax
print("\nMasked Attention Scores for First Token (all heads, before softmax):\n")
for i in range(num_heads):
    print(f"Head {i+1} Masked Attention Scores (first batch item, first query token):\n", all_head_logits[0, i, 0].numpy())

--- Task 5: Multi Head Attention ---
Input (encoded_vectors) shape: (4, 5, 4)
Multi-Head Attention Output shape: (4, 5, 4)
Multi-Head Attention Weights shape (per head, before concat): (4, 4, 5, 5)

Attention Weights for First Token (all heads):

Head 1 Attention Weights (first batch item, first query token):
 [0.99999994 0.         0.         0.         0.        ]
Head 2 Attention Weights (first batch item, first query token):
 [0.99999994 0.         0.         0.         0.        ]
Head 3 Attention Weights (first batch item, first query token):
 [0.99999994 0.         0.         0.         0.        ]
Head 4 Attention Weights (first batch item, first query token):
 [0.99999994 0.         0.         0.         0.        ]

Masked Attention Scores for First Token (all heads, before softmax):

Head 1 Masked Attention Scores (first batch item, first query token):
 [ 1.0169187e+00 -1.0000000e+09 -1.0000000e+09 -1.0000000e+09
 -1.0000000e+09]
Head 2 Masked Attention Scores (first batch i

## **Task 6: GPT Decoder Block**

Architecture:

Masked Multi Head Attention  
        ↓  
Add & Normalize  
        ↓  
Feed Forward Network  
        ↓  
Add & Normalize

Implement from scratch.

In [ ]:
import tensorflow as tf

class GPTDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, num_heads, dff, rate=0.1):
        super(GPTDecoderBlock, self).__init__()

        self.mha = MultiHeadSelfAttention(embedding_dim, num_heads)
        self.ffn = self.point_wise_feed_forward_network(embedding_dim, dff)

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def point_wise_feed_forward_network(self, embedding_dim, dff):
        return tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),  # (batch_size, seq_len, dff)
            tf.keras.layers.Dense(embedding_dim)  # (batch_size, seq_len, embedding_dim)
        ])

    def call(self, x, training, look_ahead_mask):
        # Masked Multi-Head Attention
        # x shape: (batch_size, input_seq_len, embedding_dim)
        # The `MultiHeadSelfAttention` layer expects q, k, v. For self-attention, all are x.
        attn_output, attn_weights, _ = self.mha(x, x, x, look_ahead_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)  # (batch_size, input_seq_len, embedding_dim)

        # Feed Forward Network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)  # (batch_size, input_seq_len, embedding_dim)

        return out2, attn_weights

# Define the dimensions for the Decoder Block
# embedding_dim and num_heads are already defined from previous tasks
dff = embedding_dim * 4 # Dimensionality of the inner feed-forward layer

# Instantiate the GPT Decoder Block
gpt_decoder_block = GPTDecoderBlock(embedding_dim, num_heads, dff)

# Call the decoder block with the encoded_vectors from previous steps and the look_ahead_mask
# 'training' argument is important for dropout layers
decoder_output, decoder_attn_weights = gpt_decoder_block(encoded_vectors, training=False, look_ahead_mask=look_ahead_mask)

print("--- Task 6: GPT Decoder Block ---")
print("Decoder Block Input shape:", encoded_vectors.shape)
print("Decoder Block Output shape:", decoder_output.shape)
print("Decoder Block Attention Weights shape (per head):", decoder_attn_weights.shape)
print("Decoder Block Output (first batch item):\n", decoder_output.numpy()[0])

--- Task 6: GPT Decoder Block ---
Decoder Block Input shape: (4, 5, 4)
Decoder Block Output shape: (4, 5, 4)
Decoder Block Attention Weights shape (per head): (4, 4, 5, 5)
Decoder Block Output (first batch item):
 [[ 0.7969729  -1.6690451   0.12070793  0.75136435]
 [ 1.3806717  -1.2204988  -0.623821    0.46364814]
 [ 0.6584207  -1.7156727   0.3496641   0.70758796]
 [-0.17903265 -1.5408697   1.0993614   0.6205408 ]
 [-0.8602799  -1.11147     1.1866894   0.78506035]]


## **Task 7: GPT Model**

Architecture:

Input Tokens  
      ↓  
Embedding  
      ↓  
Position Encoding  
      ↓  
Decoder Block  
      ↓  
Decoder Block  
      ↓  
Linear  
      ↓  
Softmax

In [ ]:
import tensorflow as tf

class GPTModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, max_sequence_length, num_heads, dff, num_layers, rate=0.1):
        super(GPTModel, self).__init__()

        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.max_sequence_length = max_sequence_length
        self.num_layers = num_layers

        self.decoder_blocks = [
            GPTDecoderBlock(embedding_dim, num_heads, dff, rate) for _ in range(num_layers)
        ]

        self.final_layer = tf.keras.layers.Dense(vocab_size) # Output to vocabulary size for token prediction

    def call(self, inputs, training, look_ahead_mask):
        # inputs shape: (batch_size, seq_len)

        # 1. Embedding
        x = self.embedding(inputs)  # (batch_size, seq_len, embedding_dim)

        # 2. Positional Encoding (re-create for clarity within model, or assume pre-computed)
        # For simplicity, let's re-add positional encoding here
        positional_encoding_matrix = np.zeros((self.max_sequence_length, self.embedding.output_dim))
        for pos in range(self.max_sequence_length):
            for i in range(self.embedding.output_dim):
                if i % 2 == 0:
                    positional_encoding_matrix[pos, i] = np.sin(pos / (10000 ** (i / self.embedding.output_dim)))
                else:
                    positional_encoding_matrix[pos, i] = np.cos(pos / (10000 ** ((i - 1) / self.embedding.output_dim)))
        positional_encoding_matrix = tf.constant(positional_encoding_matrix, dtype=tf.float32)
        positional_encoding_matrix = tf.expand_dims(positional_encoding_matrix, axis=0) # Add batch dim

        x += positional_encoding_matrix

        # 3. Pass through Decoder Blocks
        attention_weights_history = []
        for i in range(self.num_layers):
            x, attn_weights = self.decoder_blocks[i](x, training=training, look_ahead_mask=look_ahead_mask)
            attention_weights_history.append(attn_weights)

        # 4. Final Linear Layer
        logits = self.final_layer(x)  # (batch_size, seq_len, vocab_size)

        return logits, attention_weights_history


# Instantiate and test the GPT Model
num_layers = 2 # As suggested by the architecture diagram

gpt_model = GPTModel(vocab_size, embedding_dim, max_sequence_length, num_heads, dff, num_layers)

# Use padded_sequences (from Task 2) as input
# look_ahead_mask (from Task 5) is also needed

# Perform a forward pass
model_output_logits, all_attention_weights = gpt_model(tf.constant(padded_sequences), training=False, look_ahead_mask=look_ahead_mask)

# Apply softmax to get probabilities
model_output_probabilities = tf.nn.softmax(model_output_logits, axis=-1)

print("--- Task 7: GPT Model ---")
print("Input token IDs shape:", tf.constant(padded_sequences).shape)
print("GPT Model Output Logits shape:", model_output_logits.shape) # (batch_size, seq_len, vocab_size)
print("GPT Model Output Probabilities (first batch item, first token):")
print(model_output_probabilities.numpy()[0, 0, :])
print(f"Number of decoder blocks: {num_layers}")
print("Attention weights from first decoder block (shape per head):", all_attention_weights[0].shape)
print("Attention weights from second decoder block (shape per head):", all_attention_weights[1].shape)

--- Task 7: GPT Model ---
Input token IDs shape: (4, 5)
GPT Model Output Logits shape: (4, 5, 16)
GPT Model Output Probabilities (first batch item, first token):
[0.03004694 0.09933659 0.03776043 0.05612664 0.04545762 0.05399431
 0.02021958 0.1344801  0.05472912 0.08194946 0.09336054 0.12546964
 0.06966572 0.03455712 0.01882905 0.0440171 ]
Number of decoder blocks: 2
Attention weights from first decoder block (shape per head): (4, 4, 5, 5)
Attention weights from second decoder block (shape per head): (4, 4, 5, 5)


## **Task 8: Next Token Prediction**

Input:

- The cat chased

Prediction:

- the

* * *

Input:

- The cat chased the

Prediction:

- mouse

* * *

## **Task 9: Text Generation**

Input Prompt:

- Machine learning

Generation:

- Machine learning is  
- Machine learning is powerful  
- Machine learning is powerful because  
…

Explain:

- Autoregressive Generation

In [ ]:
def predict_next_token(model, tokenizer, input_text, max_sequence_length, temperature=1.0):
    # 1. Preprocess the input text
    input_sequence = tokenizer.texts_to_sequences([input_text])[0]
    # Pad the sequence to max_sequence_length if it's shorter
    padded_input_sequence = tf.keras.preprocessing.sequence.pad_sequences(
        [input_sequence],
        maxlen=max_sequence_length,
        padding='post'
    )
    input_ids = tf.constant(padded_input_sequence)

    # Generate a look-ahead mask for the prediction
    # The mask should be for the max_sequence_length
    seq_len = tf.shape(input_ids)[1]
    look_ahead_mask = create_look_ahead_mask(seq_len)
    look_ahead_mask = tf.expand_dims(tf.expand_dims(look_ahead_mask, axis=0), axis=0)

    # 2. Get model logits
    # Set training=False for inference
    model_output_logits, _ = model(input_ids, training=False, look_ahead_mask=look_ahead_mask)

    # The last token's logits are used for prediction
    # We need to find the actual length of the input_sequence (excluding padding)
    actual_input_length = tf.cast(tf.reduce_sum(tf.cast(tf.not_equal(input_ids, 0), tf.int32)), tf.int32)
    # If the input was padded, the prediction is for the first padded position, which is actual_input_length
    # Otherwise, it's for the last valid token
    prediction_index = actual_input_length - 1 if actual_input_length > 0 else 0
    if actual_input_length < max_sequence_length: # If there was padding, predict at the first padded spot
        prediction_index = actual_input_length
    else: # If the input filled up max_sequence_length, predict for the last token in the sequence
        prediction_index = max_sequence_length - 1

    # Get logits for the token we want to predict
    # (batch_size, seq_len, vocab_size) -> (vocab_size)
    last_token_logits = model_output_logits[0, prediction_index, :]

    # Apply temperature for sampling (optional, makes predictions more diverse)
    scaled_logits = last_token_logits / temperature

    # 3. Get probabilities and predict the next token ID
    predicted_token_id = tf.argmax(scaled_logits).numpy()

    # 4. Convert token ID back to word
    for word, index in tokenizer.word_index.items():
        if index == predicted_token_id:
            return word
    return "<unk>" # Return unknown if not found


print("--- Task 8: Next Token Prediction ---")

# Example 1: "The cat chased"
input_text_1 = "the cat chased"
predicted_word_1 = predict_next_token(gpt_model, tokenizer, input_text_1, max_sequence_length)
print(f"Input: '{input_text_1}' -> Predicted next token: '{predicted_word_1}'")

# Example 2: "The cat chased the"
input_text_2 = "the cat chased the"
predicted_word_2 = predict_next_token(gpt_model, tokenizer, input_text_2, max_sequence_length)
print(f"Input: '{input_text_2}' -> Predicted next token: '{predicted_word_2}'")

# Example 3: "Machine learning"
input_text_3 = "machine learning"
predicted_word_3 = predict_next_token(gpt_model, tokenizer, input_text_3, max_sequence_length)
print(f"Input: '{input_text_3}' -> Predicted next token: '{predicted_word_3}'")

--- Task 8: Next Token Prediction ---
Input: 'the cat chased' -> Predicted next token: 'is'
Input: 'the cat chased the' -> Predicted next token: 'barked'
Input: 'machine learning' -> Predicted next token: 'neural'


In [29]:
def generate_text(model, tokenizer, start_prompt, max_gen_length, max_sequence_length, temperature=1.0):
    generated_sequence = start_prompt.split()
    current_input_text = start_prompt

    for _ in range(max_gen_length):
        next_word = predict_next_token(model, tokenizer, current_input_text, max_sequence_length, temperature)

        if next_word == "<unk>" or next_word == '' or next_word == '[PAD]': # Stop generation if unknown or empty token or padding token is predicted
            break

        generated_sequence.append(next_word)
        current_input_text = " ".join(generated_sequence)

        # Stop if the generated sequence exceeds the model's max_sequence_length
        if len(tokenizer.texts_to_sequences([current_input_text])[0]) >= max_sequence_length:
            print(f"Stopping generation: Reached max_sequence_length ({max_sequence_length})")
            break

    return " ".join(generated_sequence)


print("--- Task 9: Text Generation ---")

# Example: "Machine learning"
input_prompt = "machine learning"
max_generation_length = 5 # Number of words to generate

print(f"Input Prompt: '{input_prompt}'")

generated_text = generate_text(gpt_model, tokenizer, input_prompt, max_generation_length, max_sequence_length)
print(f"Generated Text: '{generated_text}'")

--- Task 9: Text Generation ---
Input Prompt: 'machine learning'
Stopping generation: Reached max_sequence_length (5)
Generated Text: 'machine learning neural is barked'
